# Algebra Quick Tour

**Part I · Geometric Algebra & Core** — Tutorial 01

A rapid, example-driven tour through *pytanga*'s geometric-algebra side only —
not to master each area, but to see the big picture and know where to dive
deeper. Each section ends with a pointer to the tutorial that covers the topic
in full. For a tour of the viewer itself, see
[Part II — Visualization](../../visualization/).

> **Convention.** Algebras are created from a **basis class** (`BasisE3()`,
> `BasisPGA3()`, …) rather than the generic `Algebra(dim, sig)` constructor.
> Geometric entities and operators come from `pytanga.geometry` (via the
> `Geometry` convenience class), not from the basis classes.

## Setup

Everything used here is re-exported from a few submodules:

In [1]:
import math

from pytanga import MV
from pytanga.basis import (
    BasisE2,
    BasisE3,
    BasisN2,
    BasisN3,
    BasisP2,
    BasisPGA2,
    BasisPGA3,
)
from pytanga.geometry import (
    Direction,
    Geometry,
    Line,
    Point,
    PointPair,
    Rotor,
    Sphere,
)
from pytanga.solver.solve import solve

## 1. Algebra & Multivectors

An **algebra** is created from a basis class; a **multivector** (`MV`) is a sum
of *blades* — the scalar, vectors `e1 e2 …`, bivectors `e12 …`, and so on. Build
one from a string, then compute products and pull out grades.

→ [Tutorial 02 — Algebra & Multivectors: The Core](../02_algebra_core/)

In [2]:
E3 = BasisE3()                     # 3D Euclidean algebra G(3, 0)

a = E3("1 + 2 e1 - 3 e2")          # multivector from a string
b = E3("e1 + e2")                  # a vector

print("a        :", a)
print("a * b    :", a * b)         # geometric product
print("a ^ b    :", a ^ b)         # outer (wedge) product
print("a | b    :", a | b)         # inner product

ab = a * b
print("grade 0  :", ab.grade(0))   # scalar part
print("grade 1  :", ab.grade(1))   # vector part
print("grade 2  :", ab.grade(2))   # bivector part
print("grades   :", ab.grades)

a        : 1 + 2 e1 - 3 e2
a * b    : -1 + e1 + e2 + 5 e12
a ^ b    : e1 + e2 + 5 e12
a | b    : -1
grade 0  : -1
grade 1  : e1 + e2
grade 2  : 5 e12
grades   : [0, 1, 2]


### GA operations at a glance

| Operation | Syntax / method |
|---|---|
| Addition / subtraction | `a + b` · `a - b` |
| Negation | `-a` |
| Geometric product | `a * b` · `a.gp(b)` |
| Outer (wedge) product | `a ^ b` · `a.op(b)` |
| Inner product | `a \| b` · `a.ip(b)` |
| Scalar product | `a.sp(b)` |
| Division | `a / b` · `a / s` · `s / a` |
| Scalar scaling | `s * a` · `a * s` |
| Reverse | `~a` · `a.rev()` |
| Clifford / grade conjugate | `a.conj()` · `a.grade_conj()` |
| Multiplicative inverse | `a.inv()` |
| Versor products | `a.vp(b)` · `a.nvp(b)` |
| Grade projection | `a.grade(k)` · `a.even()` · `a.odd()` |
| Duals / complement | `a.dual()` · `a.ldual()` · `a.complement()` |
| Blade helpers | `a.blade_join(b)` · `a.blade_inverse()` · `a.blade_factorize()` |
| Project / normalize | `a.project_to(b)` · `a.normalized()` |
| Inspect | `a.components()` · `a.blade_coefs()` · `a.to_dict()` · `a.grades` |
| Cleanup | `a.prune()` |

The full reference, with every detail, is [Tutorial 02](../02_algebra_core/).

## 2. Basis Classes

*pytanga* ships eight prebuilt algebra subclasses with **named blades** as
attributes. 3D: `BasisE3`, `BasisP3`, `BasisN3` (conformal), `BasisPGA3`
(plane-based). 2D: `BasisE2`, `BasisP2`, `BasisN2`, `BasisPGA2`.

→ [Tutorial 03 — The Eight Basis Classes](../03_basis_classes/)

In [3]:
E3 = BasisE3()
PGA = BasisPGA3()

print("E3   blades:", list(E3.blades().keys()))
print("PGA3 blades:", list(PGA.blades().keys()))

print("e12 * e12  :", E3.e12 * E3.e12)   # a bivector squared → -1

# The four 2D counterparts (glimpse only)
E2, P2 = BasisE2(), BasisP2()
N2, PGA2 = BasisN2(), BasisPGA2()
print("dims       : E2=%d  P2=%d  N2=%d  PGA2=%d" % (E2.dim, P2.dim, N2.dim, PGA2.dim))

E3   blades: ['e1', 'e2', 'e3', 'e12', 'e31', 'e13', 'e23', 'I']
PGA3 blades: ['e1', 'e2', 'e3', 'ep', 'em', 'e0', 'e0_inv']
e12 * e12  : -1
dims       : E2=2  P2=3  N2=4  PGA2=4


## 3. Rotors & Motions

A **rotor** is an even-grade versor that rotates via the sandwich product
`R v ~R`. Create one through the geometry submodule.

→ [Tutorial 04 — Euclidean 3D (G3,0): Vectors, Bivectors, Rotors](../04_euclidean_e3/)

In [4]:
geo = Geometry(BasisE3())

R = geo(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
v = BasisE3()("e1")

print("R      :", R)
print("v      :", v)
print("R v ~R :", R * v * ~R)   # e1 rotated 90° about z → e2

R      : 0.7071 - 0.7071 e12
v      : e1
R v ~R : e2


## 4. Geometry Submodule

Entities (`Point`, `Line`, `Sphere`, …) are created through `pytanga.geometry`.
The `Geometry` facade binds an algebra and maps both ways with a single `geo(...)` call: `geo(entity)` creates an
`MV`; `geo(mv)` extracts the entity back out.

→ [Tutorial 14 — Geometry Submodule: Algebra-Independent Entities](../14_geometry/)

In [5]:
geo = Geometry(BasisN3())            # conformal 3D: points, lines, spheres…

mv_p = geo(Point(1, 2, 3))
print("Point  :", mv_p)
print("  →    :", geo(mv_p))

mv_l = geo(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)))
print("Line   :", geo(mv_l))

mv_s = geo(Sphere(center=Point(1, 0, 0), radius=3.0))
print("Sphere :", geo(mv_s))

Point  : e1 + 2 e2 + 3 e3 + 7 einf + eo
  →    : Point(1.00, 2.00, 3.00)
Line   : Line(org=Point(0.00, 0.00, 0.00), dir=Dir(1.00, 0.00, 0.00))
Sphere : Sphere(c=Point(1.00, -0.00, -0.00), r=3.00)


## 5. Equation Solving

Solve `A * X = B` for an unknown multivector `X` with `solve()`.

→ [Tutorial 11 — Equation Solving: From GA to Linear Systems](../11_equation_solving/)

In [6]:
E3 = BasisE3()
A = E3("1 + e12")        # a versor-like element
B = E3("e1 + e2")

X = solve(A, B, algebra=E3)
print("X     :", X)
print("A * X :", A * X)   # should equal B
print("B     :", B)

X     : e2
A * X : e1 + e2
B     : e1 + e2


## Visual Examples — Conformal N3 Entities

`pytanga.viz` renders entities straight from the conformal model (`BasisN3`).
Here, an **IPNS** scene: two spheres and their **intersection circle** (the
outer product `S1 ^ S2`), plus a **point pair**. Viewer details are deferred to
[Part II — Visualization](../../visualization/).

In [ ]:
from pytanga.viz import Visualizer, LabelStyle

# IPNS: spheres are grade-1 vectors; their outer product is the intersection circle
geo = Geometry(BasisN3(opns=False))
s1 = geo(Sphere(Point(0, 0, 0), 1.0))
s2 = geo(Sphere(Point(1.2, 0, 0), 1.0))
ci = s1 ^ s2
print("intersection circle:", geo(ci))

# OPNS point pair
pp = geo(PointPair(point_a=Point(-1, 0, 0), point_b=Point(1, 0, 0)))

viz = Visualizer(title="Quick tour — conformal N3 entities")
viz.add(s1, color="#ff4444", opacity=0.3, label="$S_1$", 
        label_style=LabelStyle(along=(0, 1,0)))
viz.add(s2, color="#4488ff", opacity=0.3, label="$S_2$")
viz.add(ci, color="#ffcc00", label=r"$S_1 \wedge S_2$")
viz.add(pp, color="#44ff44", label="point pair")

viz.display_snapshot()   # renders the scene inline (serverless)

intersection circle:

 Circle(c=Point(0.60, 0.00, 0.00), r=0.80, n=Dir(1.00, -0.00, -0.00))


## Summary & next steps

You have seen the whole algebra side at a glance:

| Area | Dive deeper |
|---|---|
| Algebra & multivectors | [02 — Algebra & Multivectors: The Core](../02_algebra_core/) |
| Basis classes | [03 — The Eight Basis Classes](../03_basis_classes/) |
| Rotors & motions | [04 — Euclidean 3D](../04_euclidean_e3/) |
| Geometry submodule | [14 — Geometry Submodule](../14_geometry/) |
| Equation solving | [11 — Equation Solving](../11_equation_solving/) |
| The viewer | [Part II — Visualization](../../visualization/) |